#4.Data Validation & Cleaning

#Check missing values

In [ ]:
print("Missing values before cleaning:")

print(
    df_extracted[
        [
            "brand_name",
            "manufacturer",
            "price_inr",
            "primary_ingredient",
            "primary_strength",
            "active_ingredients",
            "dosage_form",
            "pack_size",
            "pack_unit"
        ]
    ].isnull().sum()
)

Missing values before cleaning:
brand_name                0
manufacturer              0
price_inr                 0
primary_ingredient        0
primary_strength      25194
active_ingredients        0
dosage_form               0
pack_size             22330
pack_unit             22330
dtype: int64


#Check invalid prices

In [ ]:
invalid_prices = df_extracted[
    df_extracted["price_inr"].isna() |
    (df_extracted["price_inr"] <= 0)
]

print("Number of invalid prices:", len(invalid_prices))

Number of invalid prices: 0


In [ ]:
df_clean = df_extracted[
    df_extracted["price_inr"].notna() &
    (df_extracted["price_inr"] > 0)
].copy()

print("Records after price validation:", len(df_clean))

Records after price validation: 253969


#Check missing strength

In [ ]:
missing_strength = df_clean[
    df_clean["strength_key"] == "unknown"
]

print("Medicines with missing strength:", len(missing_strength))

Medicines with missing strength: 25194


#Remove records with unknown strength

In [ ]:
df_clean = df_clean[
    df_clean["strength_key"] != "unknown"
].copy()

print("Records after strength validation:", len(df_clean))

Records after strength validation: 228775


#Check duplicate Product IDs

In [ ]:
duplicate_ids = df_clean[
    df_clean["product_id"].duplicated(keep=False)
]

print("Duplicate product IDs:", len(duplicate_ids))

Duplicate product IDs: 0


In [ ]:
df_clean = df_clean.drop_duplicates(
    subset=["product_id"]
).copy()

print("Records after duplicate ID removal:", len(df_clean))

Records after duplicate ID removal: 228775


#Check duplicate medicine names

In [ ]:
duplicate_names = df_clean["medicine_name_key"].value_counts()

print("Medicine names appearing more than once:")

print(
    duplicate_names[
        duplicate_names > 1
    ].head(20)
)

Medicine names appearing more than once:
medicine_name_key
ringer lactate infusion              9
leflox 500mg tablet                  5
azilife 500mg tablet                 5
meditrax s 1000mg/500mg injection    4
lc 5mg tablet                        4
looz oral solution                   4
meditrax 1000mg injection            4
itracare 200mg capsule               4
zydexa plus 4mg injection            4
cefpoxim dry syrup                   4
andy 50mg injection                  3
atorfit 10 tablet                    3
vomikit 4mg tablet md                3
azbact 100mg tablet dt               3
roxim 150mg tablet                   3
top 40mg tablet                      3
oxin 200mg tablet                    3
reflox 200mg tablet                  3
sefix 100mg tablet                   3
dekadron injection                   3
Name: count, dtype: int64


#Validate comparison groups

In [ ]:
group_counts = (
    df_clean
    .groupby("comparison_group")
    .size()
    .reset_index(name="medicine_count")
)

print("Comparison group summary:")

print(group_counts.head(10))

Comparison group summary:
                      comparison_group  medicine_count
0                abacavir|300mg|tablet               4
1                abacavir|600mg|tablet               6
2               abatacept|250mg|powder               1
3             abciximab|10mg|injection               3
4             abemaciclib|100mg|tablet               1
5             abemaciclib|150mg|tablet               1
6             abemaciclib|200mg|tablet               2
7  abiraterone acetate|250mg|injection               1
8     abiraterone acetate|250mg|tablet              37
9     abiraterone acetate|500mg|tablet              14


#Identify groups containing at least 2 medicines

In [ ]:
valid_groups = group_counts[
    group_counts["medicine_count"] >= 2
]["comparison_group"]

print("Number of comparison groups with at least 2 medicines:",
      len(valid_groups))

Number of comparison groups with at least 2 medicines: 4344


#Create the comparison-ready dataset

In [ ]:
df_comparison = df_clean[
    df_clean["comparison_group"].isin(valid_groups)
].copy()

print("Comparison-ready records:", len(df_comparison))

print(
    "Comparison groups:",
    df_comparison["comparison_group"].nunique()
)

Comparison-ready records: 225731
Comparison groups: 4344


#Final validation

In [ ]:
print("FINAL DATA VALIDATION")
print("---------------------")

print("Rows:", len(df_comparison))
print("Columns:", len(df_comparison.columns))

print(
    "Unique Product IDs:",
    df_comparison["product_id"].nunique()
)

print(
    "Unique Comparison Groups:",
    df_comparison["comparison_group"].nunique()
)

print(
    "Missing Prices:",
    df_comparison["price_inr"].isna().sum()
)

print(
    "Unknown Strengths:",
    (df_comparison["strength_key"] == "unknown").sum()
)

print(
    "Duplicate Product IDs:",
    df_comparison["product_id"].duplicated().sum()
)

FINAL DATA VALIDATION
---------------------
Rows: 225731
Columns: 18
Unique Product IDs: 225731
Unique Comparison Groups: 4344
Missing Prices: 0
Unknown Strengths: 0
Duplicate Product IDs: 0
